In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime

headers = {'User-Agent': 'Mozilla/5.0 (research project)'}

# ─────────────────────────────────────────
# PHASE 1: Collect article URLs from category pages
# ─────────────────────────────────────────

def scrape_dailynews_urls(max_pages=200):
    """Daily News has clean pagination: /category/business/page/N"""
    urls = []
    categories = ['business', 'national', 'economy']
    
    for cat in categories:
        for page in range(1, max_pages + 1):
            url = f"https://dailynews.co.tz/category/{cat}/page/{page}/"
            try:
                r = requests.get(url, headers=headers, timeout=15)
                if r.status_code == 404:
                    print(f"  Reached end of {cat} at page {page}")
                    break
                    
                soup = BeautifulSoup(r.text, 'html.parser')
                # Articles are in <h2> or <h3> tags with <a> links
                links = soup.select('h2 a, h3 a, .entry-title a')
                
                for a in links:
                    href = a.get('href', '')
                    if 'dailynews.co.tz' in href:
                        urls.append(href)
                
                print(f"  Page {page} ({cat}): {len(links)} articles found")
                time.sleep(1)  # be polite
                
            except Exception as e:
                print(f"  Error on page {page}: {e}")
                break
    
    return list(set(urls))  # deduplicate


def scrape_citizen_urls(max_pages=200):
    """The Citizen: /tanzania/news/[category]?page=N"""
    urls = []
    categories = ['business', 'national', 'economy']
    
    for cat in categories:
        for page in range(1, max_pages + 1):
            url = f"https://www.thecitizen.co.tz/tanzania/news/{cat}?page={page}"
            try:
                r = requests.get(url, headers=headers, timeout=15)
                if r.status_code != 200:
                    print(f"  Stopped at page {page} for {cat}")
                    break
                
                soup = BeautifulSoup(r.text, 'html.parser')
                links = soup.select('h2 a, h3 a, .article-title a, .story-title a')
                
                for a in links:
                    href = a.get('href', '')
                    if href and not href.startswith('http'):
                        href = 'https://www.thecitizen.co.tz' + href
                    if 'thecitizen.co.tz' in href:
                        urls.append(href)
                
                print(f"  Page {page} ({cat}): {len(links)} found")
                time.sleep(1.5)
                
            except Exception as e:
                print(f"  Error: {e}")
                break
    
    return list(set(urls))


# ─────────────────────────────────────────
# PHASE 2: Scrape each article for headline + date
# ─────────────────────────────────────────

def scrape_article(url, source):
    try:
        r = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(r.text, 'html.parser')
        
        # Headline — try common selectors
        headline = None
        for sel in ['h1.article-title', 'h1.entry-title', 'h1.post-title', 'h1']:
            tag = soup.select_one(sel)
            if tag:
                headline = tag.get_text(strip=True)
                break
        
        # Date — look for <time> tag or meta tag
        date = None
        time_tag = soup.find('time')
        if time_tag:
            date = time_tag.get('datetime') or time_tag.get_text(strip=True)
        
        if not date:
            meta = soup.find('meta', {'property': 'article:published_time'})
            if meta:
                date = meta.get('content')
        
        return {'headline': headline, 'date': date, 'url': url, 'source': source}
    
    except Exception as e:
        return None


# ─────────────────────────────────────────
# RUN IT — start small to test
# ─────────────────────────────────────────



In [2]:
# Test with just 3 pages first
print("=== Collecting Daily News URLs ===")
dn_urls = scrape_dailynews_urls(max_pages=3)
print(f"\nTotal URLs collected: {len(dn_urls)}")

=== Collecting Daily News URLs ===
  Page 1 (business): 20 articles found
  Page 2 (business): 20 articles found
  Page 3 (business): 20 articles found
  Reached end of national at page 1
  Page 1 (economy): 10 articles found
  Page 2 (economy): 10 articles found
  Page 3 (economy): 10 articles found

Total URLs collected: 53


In [3]:
# Scrape the actual articles
print("\n=== Scraping article details ===")
records = []
for url in dn_urls[:20]:  # test on 20 first
    article = scrape_article(url, source='Daily News')
    if article and article['headline']:
        records.append(article)
        print(f"✓ {article['date']} | {article['headline'][:60]}")
    time.sleep(1)

df = pd.DataFrame(records)
print(f"\n{len(df)} articles scraped successfully")
print(df.head())


=== Scraping article details ===
✓ 2026-05-14T07:11:05+00:00 | Carbon trading unlocks opportunities
✓ None | Error establishing a database connection
✓ 2026-05-13T07:48:13+00:00 | Institutional conflicts demand skilled, responsible leadersh
✓ 2026-05-05T11:40:41+00:00 | Ruto banks on the intra-regional trade as key to advance the
✓ 2026-05-13T09:21:33+00:00 | AfDB meetings to focus on Africa financing gap
✓ 2026-05-13T05:15:08+00:00 | Africa eyes shared prosperity
✓ 2026-05-17T18:26:14+03:00 | EACOP advances toward final stages
✓ 2026-05-13T09:42:22+00:00 | EADB launches USD 13M fund against EAC unemployment
✓ None | Error establishing a database connection
✓ 2026-04-20T04:09:06+03:00 | VISION 2050: Clear path toward a trillion-dollar economy by 
✓ 2026-05-14T10:58:40+00:00 | Government backs Private Sector to drive Vision 2050 goals
✓ None | Error establishing a database connection
✓ 2026-05-13T07:40:52+00:00 | Visa users compete for luxury Mauritius trip
✓ 2026-04-16T23:00:29+03:00 

In [6]:
df

,headline,date,url,source
0,Carbon trading unlocks opportunities,2026-05-14T07:11:05+00:00,https://dailynews.co.tz/carbon-trading-unlocks...,Daily News
1,Error establishing a database connection,None,https://dailynews.co.tz/collective-schemes-enh...,Daily News
2,"Institutional conflicts demand skilled, respon...",2026-05-13T07:48:13+00:00,https://dailynews.co.tz/institutional-conflict...,Daily News
3,Ruto banks on the intra-regional trade as key ...,2026-05-05T11:40:41+00:00,https://dailynews.co.tz/ruto-banks-on-the-intr...,Daily News
4,AfDB meetings to focus on Africa financing gap,2026-05-13T09:21:33+00:00,https://dailynews.co.tz/afdb-meetings-to-focus...,Daily News
5,Africa eyes shared prosperity,2026-05-13T05:15:08+00:00,https://dailynews.co.tz/africa-eyes-shared-pro...,Daily News
6,EACOP advances toward final stages,2026-05-17T18:26:14+03:00,https://dailynews.co.tz/eacop-advances-toward-...,Daily News
7,EADB launches USD 13M fund against EAC unemplo...,2026-05-13T09:42:22+00:00,https://dailynews.co.tz/eadb-launches-usd-13m-...,Daily News
8,Error establishing a database connection,None,https://dailynews.co.tz/banks-urged-to-deepen-...,Daily News
9,VISION 2050: Clear path toward a trillion-doll...,2026-04-20T04:09:06+03:00,https://dailynews.co.tz/vision-2050-clear-path...,Daily News


In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {'User-Agent': 'Mozilla/5.0 (research project)'}

def scrape_article(url, source):
    try:
        r = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(r.text, 'html.parser')
        
        # Skip WordPress database error pages
        page_text = soup.get_text()
        if 'Error establishing a database connection' in page_text:
            return None
        
        # Headline
        headline = None
        for sel in ['h1.article-title', 'h1.entry-title', 'h1.post-title', 'h1']:
            tag = soup.select_one(sel)
            if tag:
                headline = tag.get_text(strip=True)
                break
        
        # Date
        date = None
        time_tag = soup.find('time')
        if time_tag:
            date = time_tag.get('datetime') or time_tag.get_text(strip=True)
        if not date:
            meta = soup.find('meta', {'property': 'article:published_time'})
            if meta:
                date = meta.get('content')
        
        if not headline:
            return None
            
        return {'headline': headline, 'date': date, 'url': url, 'source': source}
    
    except Exception as e:
        return None

def scrape_dailynews_range(start_page, end_page):
    categories = ['business']
    all_records = []
    
    for cat in categories:
        print(f"\n── Category: {cat} ──")
        for page in range(start_page, end_page + 1):
            if page % 20 == 0:
                pd.DataFrame(all_records).to_csv('checkpoint.csv', index=False)
                print(f"  [Checkpoint saved — {len(all_records)} records so far]")
            
            url = f"https://dailynews.co.tz/category/{cat}/page/{page}/"
            try:
                r = requests.get(url, headers=headers, timeout=15)
                if r.status_code == 404:
                    print(f"  End of {cat} at page {page}")
                    break
                
                soup = BeautifulSoup(r.text, 'html.parser')
                
                # ── FIXED: only grab links inside the masonry archive grid ──
                masonry = soup.select_one('#masonry-grid')
                if not masonry:
                    print(f"  Page {page}: masonry grid not found, skipping")
                    continue
                
                links = masonry.select('h2.entry-title a')
                article_urls = [a['href'] for a in links if a.get('href')]
                
                print(f"  Page {page}: {len(article_urls)} articles", end=' → ')
                
                page_records = []
                for art_url in article_urls:
                    article = scrape_article(art_url, 'Daily News')
                    if article:
                        page_records.append(article)
                    time.sleep(0.5)
                
                # Checkpoint every 20 pages
                if page % 20 == 0:
                    pd.DataFrame(all_records).to_csv('checkpoint.csv', index=False)
                    print(f"\n  [Checkpoint saved — {len(all_records)} records total]", end='')
                
                all_records.extend(page_records)
                print(f" {len(page_records)} valid")
                time.sleep(1)
                
            except Exception as e:
                print(f"  Error on page {page}: {e}")
                continue
    
    return all_records


# ── Filter to only 2022–2024 ──
def filter_by_year(records, years=[2022, 2023, 2024]):
    filtered = []
    for r in records:
        if r['date']:
            try:
                year = int(str(r['date'])[:4])
                if year in years:
                    filtered.append(r)
            except:
                pass
    return filtered

In [6]:
print("=== Full scrape: pages 287-683 (2022-2024) ===")
full_records = scrape_dailynews_range(start_page=600, end_page=683)

# Filter to strictly 2022-2024 just in case of overlap
filtered = filter_by_year(full_records, years=[2022, 2023, 2024])

df_final = pd.DataFrame(filtered)
print(f"\nTotal articles collected: {len(df_final)}")
print(df_final['year'].value_counts())

# Save
df_final.to_csv('dailynews_headlines_2022_2024.csv', index=False)
print("Saved to dailynews_headlines_2022_2024.csv")

=== Full scrape: pages 287-683 (2022-2024) ===

── Category: business ──
  [Checkpoint saved — 0 records so far]
  Page 600: 10 articles → 
  [Checkpoint saved — 0 records total] 6 valid
  Page 601: masonry grid not found, skipping
  Page 602: masonry grid not found, skipping
  Page 603: 10 articles →  2 valid
  Page 604: masonry grid not found, skipping
  Page 605: 10 articles →  6 valid
  Page 606: 10 articles →  8 valid
  Error on page 607: HTTPSConnectionPool(host='dailynews.co.tz', port=443): Read timed out. (read timeout=15)
  Error on page 608: HTTPSConnectionPool(host='dailynews.co.tz', port=443): Read timed out. (read timeout=15)
  Page 609: masonry grid not found, skipping
  Error on page 610: HTTPSConnectionPool(host='dailynews.co.tz', port=443): Read timed out. (read timeout=15)
  Error on page 611: HTTPSConnectionPool(host='dailynews.co.tz', port=443): Read timed out. (read timeout=15)
  Page 612: 10 articles →  2 valid
  Page 613: 10 articles →  5 valid
  Page 614: masonr

KeyError: 'year'

In [ ]:
start ; 287
end : 683